# Pneumonia Detection — 02: Training & Evaluation

Loads preprocessed arrays from `01_eda_preprocessing.ipynb`, trains a VGG16 transfer-learning model with class-weighted loss, and evaluates with confusion matrix, sensitivity/specificity, AUC-ROC, learning curves, and Grad-CAM explainability.

## Step 1: Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import cv2
import tensorflow as tf
import keras

from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.inception_v3 import InceptionV3

import warnings
warnings.filterwarnings("ignore")

## Step 2: Load preprocessed data

Loads the arrays saved at the end of `01_eda_preprocessing.ipynb` — instant, no need to re-run `get_data()`.

In [ ]:
# Match SAVE_DIR to wherever notebook 1 saved the files
# from google.colab import drive
# drive.mount('/content/drive')
# SAVE_DIR = '/content/drive/MyDrive/pneumonia_project/'

SAVE_DIR = './'

X_train = np.load(SAVE_DIR + 'X_train.npy')
y_train = np.load(SAVE_DIR + 'y_train.npy')
X_test = np.load(SAVE_DIR + 'X_test.npy')
y_test = np.load(SAVE_DIR + 'y_test.npy')

Y_trainHot = to_categorical(y_train, num_classes=2)
Y_testHot = to_categorical(y_test, num_classes=2)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

## Step 3: Class weights & pretrained models

Uses `class_weight='balanced'` instead of resampling, so no training data is discarded — errors on the minority class (Normal) are penalized more heavily during training.

In [ ]:
map_characters1 = {0: 'No Pneumonia', 1: 'Yes Pneumonia'}

raw_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight1 = dict(enumerate(raw_weights))
print("Class weights:", class_weight1)

# Let Keras download the weights directly via 'imagenet'
pretrained_model_1 = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
pretrained_model_2 = InceptionV3(weights='imagenet', include_top=False, input_shape=(150, 150, 3))

optimizer1 = keras.optimizers.RMSprop(learning_rate=0.0001)

## Step 4: Define the transfer-learning network

VGG16 backbone (frozen) + light medical-safe augmentation + a new classification head. Returns both the trained model and its training history.

In [ ]:
def pretrainedNetwork(xtrain, ytrain, xtest, ytest, pretrainedmodel,
                       classweight, numclasses, numepochs, optimizer, labels):
    base_model = pretrainedmodel

    # Safe medical data augmentation — small, conservative transforms only
    data_augmentation = keras.Sequential([
        keras.layers.RandomRotation(0.03, fill_mode='constant', fill_value=0.0),
        keras.layers.RandomZoom(0.1, fill_mode='constant', fill_value=0.0),
        keras.layers.RandomTranslation(0.1, 0.1, fill_mode='constant', fill_value=0.0)
    ], name="medical_data_augmentation")

    inputs = Input(shape=(150, 150, 3))
    x = data_augmentation(inputs)          # only active during training
    x = base_model(x, training=False)      # frozen backbone, locked BatchNorm stats

    x = Flatten()(x)
    predictions = Dense(numclasses, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=predictions)

    for layer in base_model.layers:
        layer.trainable = False

    model.compile(loss='categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])

    callbacks_list = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, verbose=1)]

    model.summary()

    history = model.fit(xtrain, ytrain,
                        epochs=numepochs,
                        class_weight=classweight,
                        validation_data=(xtest, ytest),
                        verbose=1,
                        callbacks=callbacks_list)

    score = model.evaluate(xtest, ytest, verbose=0)
    print('\nKeras CNN - Test accuracy:', score[1], '\n')

    y_pred = model.predict(xtest)
    print('\nClassification Report:\n',
          sklearn.metrics.classification_report(np.argmax(ytest, axis=1),
                                                np.argmax(y_pred, axis=1),
                                                target_names=list(labels.values())))

    return model, history

## Step 5: Train the model

In [ ]:
model, history = pretrainedNetwork(
    xtrain=X_train,
    ytrain=Y_trainHot,
    xtest=X_test,
    ytest=Y_testHot,
    pretrainedmodel=pretrained_model_1,
    classweight=class_weight1,
    numclasses=2,
    numepochs=30,
    optimizer=optimizer1,
    labels=map_characters1
)

## Step 6: Evaluation — confusion matrix, sensitivity, specificity, AUC-ROC

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred_labels = np.argmax(y_pred_probs, axis=1)
y_true_labels = y_test
y_pred_pneumonia_prob = y_pred_probs[:, 1]

cm = confusion_matrix(y_true_labels, y_pred_labels)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)
print(f"\nTrue Negatives  (correctly No Pneumonia): {tn}")
print(f"False Positives (false alarm):              {fp}")
print(f"False Negatives (missed Pneumonia):         {fn}  \u26a0 most critical")
print(f"True Positives  (correctly caught):         {tp}")

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Pneumonia', 'Yes Pneumonia'],
            yticklabels=['No Pneumonia', 'Yes Pneumonia'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100)
plt.show()

In [ ]:
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)
f1 = 2 * (precision * sensitivity) / (precision + sensitivity)

print(f"\nAccuracy:    {accuracy:.4f}")
print(f"Sensitivity: {sensitivity:.4f}  (catches {sensitivity*100:.1f}% of real Pneumonia cases)")
print(f"Specificity: {specificity:.4f}  (correctly clears {specificity*100:.1f}% of healthy patients)")
print(f"Precision:   {precision:.4f}")
print(f"F1 Score:    {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_true_labels, y_pred_labels,
                            target_names=['No Pneumonia', 'Yes Pneumonia']))

In [ ]:
fpr, tpr, thresholds = roc_curve(y_true_labels, y_pred_pneumonia_prob)
auc_score = roc_auc_score(y_true_labels, y_pred_pneumonia_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#1D9E75', linewidth=2, label=f'ROC curve (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1, label='Random guess (AUC = 0.5)')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve \u2014 Pneumonia Detection')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=100)
plt.show()

print(f"\nAUC-ROC Score: {auc_score:.4f}")

## Step 7: Learning curves — check for overfitting

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Accuracy over epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss over epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=100)
plt.show()

## Step 8: Grad-CAM explainability

Builds a flat (non-nested) version of the model sharing the same trained weights, so Grad-CAM gradients can be traced through a single clean graph.

In [ ]:
vgg_submodel = model.get_layer('vgg16')
flat_input = vgg_submodel.input
flat_vgg_output = vgg_submodel.output

# Keep only the classifier head layers (Flatten, Dense) — skip augmentation
extra_layers = [l for l in model.layers if l.name in ['flatten_6', 'dense_6']]
print("Layers after VGG16:", [l.name for l in extra_layers])

x = flat_vgg_output
for layer in extra_layers:
    x = layer(x)

flat_model = Model(inputs=flat_input, outputs=x)

grad_model = Model(
    inputs=flat_model.input,
    outputs=[flat_model.get_layer('block5_conv3').output, flat_model.output]
)

def make_gradcam_heatmap(img_array, pred_index=None):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)


def overlay_gradcam(img, heatmap, alpha=0.4):
    img_uint8 = np.uint8(255 * img) if img.max() <= 1.0 else np.uint8(img)
    heatmap_resized = cv2.resize(heatmap, (img_uint8.shape[1], img_uint8.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img_uint8, 1 - alpha, heatmap_colored, alpha, 0)

In [ ]:
class_names = ['No Pneumonia', 'Yes Pneumonia']
indices = np.random.choice(len(X_test), 6, replace=False)

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle("Grad-CAM: Where the model looks when predicting", fontsize=14)

for col, idx in enumerate(indices):
    img = X_test[idx]
    img_batch = np.expand_dims(img.astype('float32'), axis=0)

    heatmap, pred_class = make_gradcam_heatmap(img_batch)
    overlaid = overlay_gradcam(img, heatmap)

    true_class = int(y_test[idx])
    correct = "\u2713" if pred_class == true_class else "\u2717"

    axes[0, col].imshow(img)
    axes[0, col].set_title(f"True: {class_names[true_class]}", fontsize=9)
    axes[0, col].axis('off')

    axes[1, col].imshow(overlaid)
    axes[1, col].set_title(f"{correct} Pred: {class_names[pred_class]}", fontsize=9,
                            color='green' if correct == "\u2713" else 'red')
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig('gradcam_examples.png', dpi=100)
plt.show()
print("Saved: gradcam_examples.png")

## Step 9: Save model and results

In [ ]:
import pickle
import json

model.save('pneumonia_model.keras')

with open('training_history.pkl', 'wb') as f:
    pickle.dump(history.history, f)

results_summary = {
    "accuracy": float(accuracy),
    "sensitivity": float(sensitivity),
    "specificity": float(specificity),
    "precision": float(precision),
    "f1_score": float(f1),
    "auc_roc": float(auc_score),
    "confusion_matrix": {
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp)
    }
}

with open('results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print(json.dumps(results_summary, indent=2))